### Sampling Tests

**Tests cover:** structural correctness, reproducibility, idempotency, and validation errors.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from dahlia.datasets.na_generator import ExperimentSampler, PredefinedSetProvider

iris = load_iris(as_frame=True)
df = iris.data.rename(columns={
    "sepal length (cm)": "SepalLengthCm",
    "sepal width (cm)": "SepalWidthCm",
    "petal length (cm)": "PetalLengthCm",
    "petal width (cm)": "PetalWidthCm",
})

print(f"Dataset loaded: {df.shape}")

Dataset loaded: (150, 4)


In [5]:
def expect_error(error_type, fn, *args, **kwargs):
    """Asserts that fn(*args, **kwargs) raises error_type."""
    try:
        fn(*args, **kwargs)
        print(f"  [FAIL] Expected {error_type.__name__}, got no error")
    except error_type as e:
        print(f"  [OK]   {error_type.__name__}: {e}")
    except Exception as e:
        print(f"  [FAIL] Expected {error_type.__name__}, got {type(e).__name__}: {e}")

### 1. ExperimentSampler Tests

In [3]:
sampler = ExperimentSampler(seed=42)
na_idx, imp_idx = sampler.select_experiment_points(df, na_fraction=0.2)

print(f"NA fraction 0.2 → {len(na_idx)} removed points, {len(imp_idx)} imputation steps")
print(f"na_indices      : {na_idx}")
print(f"imp_indices     : {imp_idx}")

NA fraction 0.2 → 30 removed points, 15 imputation steps
na_indices      : [ 10  12  17  25  26  52  54  59  63  66  68  70  80  81  89  94  96  97
 101 106 108 113 116 121 124 126 128 133 140 145]
imp_indices     : [ 12  63  68 113 108  52  94  54 121  25 140  81 106  59  80]


#### Idempotency: same instance, multiple calls must return identical results

In [6]:
sampler = ExperimentSampler(seed=42)
na1, imp1 = sampler.select_experiment_points(df, na_fraction=0.2)
na2, imp2 = sampler.select_experiment_points(df, na_fraction=0.2)

assert np.array_equal(na1, na2), "FAIL: na_indices differ between calls on same instance"
assert np.array_equal(imp1, imp2), "FAIL: imp_indices differ between calls on same instance"
print("[OK] Same instance, two calls -> identical results (idempotent)")


[OK] Same instance, two calls -> identical results (idempotent)


#### Reproducibility: two separate instances with the same seed

In [7]:
sampler_a = ExperimentSampler(seed=42)
sampler_b = ExperimentSampler(seed=42)

na_a, imp_a = sampler_a.select_experiment_points(df, na_fraction=0.2)
na_b, imp_b = sampler_b.select_experiment_points(df, na_fraction=0.2)

assert np.array_equal(na_a, na_b), "FAIL: different instances, same seed -> different na_indices"
assert np.array_equal(imp_a, imp_b), "FAIL: different instances, same seed -> different imp_indices"
print("[OK] Two separate instances, same seed -> identical results")


[OK] Two separate instances, same seed -> identical results


#### Isolation: different seeds must give different results

In [8]:
sampler_0 = ExperimentSampler(seed=0)
sampler_1 = ExperimentSampler(seed=1)

na_0, imp_0 = sampler_0.select_experiment_points(df, na_fraction=0.2)
na_1, imp_1 = sampler_1.select_experiment_points(df, na_fraction=0.2)

assert not np.array_equal(na_0, na_1), "FAIL: different seeds produced identical na_indices"
assert not np.array_equal(imp_0, imp_1), "FAIL: different seeds produced identical imp_indices"
print("[OK] Different seeds -> different results")


[OK] Different seeds -> different results


#### Structural correctness

In [9]:
sampler = ExperimentSampler(seed=99)
na_idx, imp_idx = sampler.select_experiment_points(df, na_fraction=0.3, n_imputation_steps=10)

expected_na = int(len(df) * 0.3)
assert len(na_idx) == expected_na, f"FAIL: expected {expected_na} NA points, got {len(na_idx)}"
assert len(imp_idx) == 10, f"FAIL: expected 10 imputation steps, got {len(imp_idx)}"
print(f"[OK] Correct counts: {len(na_idx)} NA points, {len(imp_idx)} imputation steps")

assert list(na_idx) == sorted(na_idx), "FAIL: na_indices not sorted"
print("[OK] na_indices are sorted")

assert set(imp_idx).issubset(set(na_idx)), "FAIL: imp_indices not subset of na_indices"
print("[OK] imputation_indices is a subset of na_indices")

assert len(set(na_idx)) == len(na_idx), "FAIL: duplicates in na_indices"
assert len(set(imp_idx)) == len(imp_idx), "FAIL: duplicates in imputation_indices"
print("[OK] No duplicates in either result")

[OK] Correct counts: 45 NA points, 10 imputation steps
[OK] na_indices are sorted
[OK] imputation_indices is a subset of na_indices
[OK] No duplicates in either result


#### Boundary: n_imputation_steps exactly equal to n_na

In [17]:
n_na_exact = int(len(df) * 0.1)
sampler = ExperimentSampler(seed=0)
na_idx, imp_idx = sampler.select_experiment_points(
    df, na_fraction=0.1, n_imputation_steps=n_na_exact
)
assert len(imp_idx) == n_na_exact
assert set(imp_idx) == set(na_idx)
print(f"[OK] n_imputation_steps == n_na ({n_na_exact}) is valid")

[OK] n_imputation_steps == n_na (15) is valid


#### Non-contiguous index (e.g. filtered DataFrame with index gaps)

In [18]:
# Indices: 0, 2, 4, 6, ..., 148
df_filtered = df.iloc[::2].copy()   
sampler_a = ExperimentSampler(seed=7)
sampler_b = ExperimentSampler(seed=7)

na_a, imp_a = sampler_a.select_experiment_points(df_filtered, na_fraction=0.2)
na_b, imp_b = sampler_b.select_experiment_points(df_filtered, na_fraction=0.2)

assert np.array_equal(na_a, na_b), "FAIL: non-contiguous index - not reproducible"
assert all(idx in df_filtered.index for idx in na_a), "FAIL: na_indices contain invalid index values"
assert all(idx in df_filtered.index for idx in imp_a), "FAIL: imp_indices contain invalid index values"
print("[OK] Non-contiguous DataFrame index handled correctly")


[OK] Non-contiguous DataFrame index handled correctly


#### Reproducibility across all NA fraction / step count combinations

In [10]:
SEED = 42
configs = [
    (0.1, 15),
    (0.2, 15),
    (0.3, 15),
    (0.2, 5),
    (0.2, 10),
]

for na_frac, n_steps in configs:
    na_a, imp_a = ExperimentSampler(seed=SEED).select_experiment_points(df, na_frac, n_steps)
    na_b, imp_b = ExperimentSampler(seed=SEED).select_experiment_points(df, na_frac, n_steps)

    assert np.array_equal(na_a, na_b), f"FAIL: not reproducible for NA={na_frac}, steps={n_steps}"
    assert np.array_equal(imp_a, imp_b), f"FAIL: not reproducible for NA={na_frac}, steps={n_steps}"
    print(f"[OK] NA={na_frac}, steps={n_steps} -> reproducible")

[OK] NA=0.1, steps=15 -> reproducible
[OK] NA=0.2, steps=15 -> reproducible
[OK] NA=0.3, steps=15 -> reproducible
[OK] NA=0.2, steps=5 -> reproducible
[OK] NA=0.2, steps=10 -> reproducible


#### Validation errors: ExperimentSampler seed

In [11]:
print("--- ExperimentSampler seed validation ---")
expect_error(TypeError,  ExperimentSampler, True)
expect_error(TypeError,  ExperimentSampler, False)
expect_error(TypeError,  ExperimentSampler, 3.14)
expect_error(TypeError,  ExperimentSampler, "42")
expect_error(ValueError, ExperimentSampler, -1)

--- ExperimentSampler seed validation ---
  [OK]   TypeError: seed must be a non-negative int, got bool
  [OK]   TypeError: seed must be a non-negative int, got bool
  [OK]   TypeError: seed must be a non-negative int, got float
  [OK]   TypeError: seed must be a non-negative int, got str
  [OK]   ValueError: seed must be >= 0, got -1


#### Validation errors: select_experiment_points na_fraction

In [12]:
sampler = ExperimentSampler(seed=0)

print("--- na_fraction validation ---")
expect_error(ValueError, sampler.select_experiment_points, df, na_fraction=0.0)
expect_error(ValueError, sampler.select_experiment_points, df, na_fraction=1.0)
expect_error(ValueError, sampler.select_experiment_points, df, na_fraction=-0.1)
expect_error(ValueError, sampler.select_experiment_points, df, na_fraction=1.5)

--- na_fraction validation ---
  [OK]   ValueError: na_fraction must be in (0.0, 1.0), got 0.0
  [OK]   ValueError: na_fraction must be in (0.0, 1.0), got 1.0
  [OK]   ValueError: na_fraction must be in (0.0, 1.0), got -0.1
  [OK]   ValueError: na_fraction must be in (0.0, 1.0), got 1.5


#### Validation errors: select_experiment_points n_imputation_steps

In [14]:
sampler = ExperimentSampler(seed=0)

print("--- n_imputation_steps validation ---")
expect_error(TypeError,  sampler.select_experiment_points, df, n_imputation_steps=True)
expect_error(TypeError,  sampler.select_experiment_points, df, n_imputation_steps=5.0)
expect_error(ValueError, sampler.select_experiment_points, df, n_imputation_steps=0)
expect_error(ValueError, sampler.select_experiment_points, df, n_imputation_steps=-1)
expect_error(ValueError, sampler.select_experiment_points, df, na_fraction=0.1,
             n_imputation_steps=16)

--- n_imputation_steps validation ---
  [OK]   TypeError: n_imputation_steps must be int, got bool
  [OK]   TypeError: n_imputation_steps must be int, got float
  [OK]   ValueError: n_imputation_steps must be >= 1, got 0
  [OK]   ValueError: n_imputation_steps must be >= 1, got -1
  [OK]   ValueError: Cannot select 16 imputation steps from 15 NA points.


### 2. PredefinedSetProvider Tests

#### PredefinedSetProvider: indices correctness + order preservation

In [15]:
na_idx, imp_idx = ExperimentSampler(seed=0).select_experiment_points(df, 0.2)

predefined_data = {
    "removed_indices": na_idx.tolist(),
    "imputed_indices": imp_idx.tolist(),
}

provider = PredefinedSetProvider(predefined_data)
ret_na, ret_imp = provider.get_experiment_points()

assert np.array_equal(ret_na, na_idx), "FAIL: returned na_indices do not match input"
assert np.array_equal(ret_imp, imp_idx), "FAIL: returned imp_indices do not match input"
print("[OK] PredefinedSetProvider returns correct indices")

assert list(ret_imp) == predefined_data["imputed_indices"], "FAIL: imputation order was changed"
print("[OK] Imputation order preserved exactly as stored")

[OK] PredefinedSetProvider returns correct indices
[OK] Imputation order preserved exactly as stored


#### PredefinedSetProvider: empty lists

In [20]:
provider_empty = PredefinedSetProvider({
    "removed_indices": [],
    "imputed_indices": [],
})
na_empty, imp_empty = provider_empty.get_experiment_points()
assert len(na_empty) == 0
assert len(imp_empty) == 0
print("[OK] PredefinedSetProvider accepts empty lists")


[OK] PredefinedSetProvider accepts empty lists


#### PredefinedSetProvider: validation errors

In [16]:
valid_removed = [10, 20, 30, 40, 50]
valid_imputed = [10, 20, 30]

print("--- missing keys ---")
expect_error(KeyError, PredefinedSetProvider, {"removed_indices": valid_removed})
expect_error(KeyError, PredefinedSetProvider, {"imputed_indices": valid_imputed})
expect_error(KeyError, PredefinedSetProvider, {})

print("\n--- duplicates ---")
expect_error(ValueError, PredefinedSetProvider, {
    "removed_indices": [10, 10, 20, 30],
    "imputed_indices": [10],
})
expect_error(ValueError, PredefinedSetProvider, {
    "removed_indices": valid_removed,
    "imputed_indices": [10, 10, 20],
})

print("\n--- imputed not subset of removed ---")
expect_error(ValueError, PredefinedSetProvider, {
    "removed_indices": valid_removed,
    "imputed_indices": [10, 20, 999],
})

--- missing keys ---
  [OK]   KeyError: "predefined_set_data is missing required keys: {'imputed_indices'}"
  [OK]   KeyError: "predefined_set_data is missing required keys: {'removed_indices'}"
  [OK]   KeyError: "predefined_set_data is missing required keys: {'removed_indices', 'imputed_indices'}"

--- duplicates ---
  [OK]   ValueError: removed_indices contains duplicate entries.
  [OK]   ValueError: imputed_indices contains duplicate entries.

--- imputed not subset of removed ---
  [OK]   ValueError: imputed_indices contains entries not present in removed_indices: [999]


### All tests passed.